## Parte 1: Comunicación distribuida

Se encontraron distintos obstáculos para la comunicación distribuida en ROS, particularmente si la red se encuentra restringida como la red de la universidad, será imposible conectar los equipos. Por tanto, se recurrió al hotspot wifi de un teléfono android y a la demostración por medio del uso de una máquina virtual dentro de la cual se corrió otra instancia de ROS1 y que fue conectada al host en la modalidad de red "bridge" con el fin de poder establecer la conexión. Estas dos alternativas funcionaron sin mayores contratiempos.

Otra alternativa sugerida, especialmente si se pretende trabajar via internet y la red local se encuentra detrás de un NAT, es el uso de una VPN gratuita como Tailscale.   

En el siguiente  se demuestra la comunicación distribuida, mediante el método de la máquina virtual (VM). Dentro de cada una de las máquinas, vm y host, se estable la variable de entorno ROS_IP como la IP de cada máquina. Posteriormente se inicializa _roscore_ dentro de la máquina virtual. Es importante resaltar que durante la misma inicialización el programa mostrará cuál es el valor de ROS_MASTER_URI, parámetro que debe pasarse a las otras máquinas mediante el comando _export_.

Una vez hecho esto, se inicia el nodo turtlesim en la máquina virtual. Simultáneamente en la máquina host, se inicia el nodo de teleoperación de la tortuga. De esta manera, la operación de la tortuga con el teclado se realiza desde otra máquina distinta via red.

![](img/lab2_distributed_demo.gif)

Posteriormente, 

## Parte 2: Creación de un nodo propio

Se escribió un nodo en Python para controlar la tortuga de forma autónoma. El diagrama de flujo correspondiente al mismo se deja a continuación:

![](img/lab2_flowchart.drawio.png)

Por estar escrito en Python este debe ubicarse dentro de la carpeta src/autoturtle/scripts en el directorio $catkin_ws y se debe añadir a la lista de compilación para que catkin make lo incluya. A continuación se muestran el código en Python del script del nodo y el package.xml, junto con un video demostrativo.

### autoturtle.py


```python
#!/usr/bin/env python3
import rospy
from geometry_msgs.msg import Twist
from turtlesim.msg import Pose
from turtlesim.srv import SetPen

class TurtleController:
    def __init__(self):
        # 1. Inicializar el nodo
        rospy.init_node('autoturtle', anonymous=True)

        # 2. Publisher: Comandos de velocidad
        self.velocity_publisher = rospy.Publisher('/turtle1/cmd_vel', Twist, queue_size=10)

        # 3. Subscriber: Posición de la tortuga
        self.pose_subscriber = rospy.Subscriber('/turtle1/pose', Pose, self.update_pose)

        # 4. Servicie client: Cambiar el color de la pluma
        rospy.wait_for_service('/turtle1/set_pen')
        self.set_pen_service = rospy.ServiceProxy('/turtle1/set_pen', SetPen)

        self.pose = Pose()
        self.rate = rospy.Rate(10) # 10 Hz
        self.current_side = "left"

    def update_pose(self, data):
        """Callback que se ejecuta cada vez que recibimos la posición."""
        self.pose = data
        self.check_pen_color()

    def check_pen_color(self):
        """Cambia el color si cruza x = 5.5"""
        if self.pose.x > 5.5 and self.current_side == "left":
            self.set_pen_service(255, 0, 0, 3, 0) # R
            self.current_side = "right"
        elif self.pose.x <= 5.5 and self.current_side == "right":
            self.set_pen_service(0, 255, 0, 3, 0) # G
            self.current_side = "left"

    def move(self):
        """Mover la tortuga y evitar los bordes"""
        vel_msg = Twist()
        while not rospy.is_shutdown():
            # Límites de turtlesim son 0.0 a 11.0
            if (self.pose.x < 1 or self.pose.x > 10 or 
                self.pose.y < 1 or self.pose.y > 10):
            # Cerca del borde: Girar
                vel_msg.linear.x = 1.0
                vel_msg.angular.z = 2
            else:
                vel_msg.linear.x = 2.0
                vel_msg.angular.z = 0.0

            self.velocity_publisher.publish(vel_msg)
            self.rate.sleep()

if __name__ == '__main__':
    try:
        controller = TurtleController()
        controller.move()
    except rospy.ROSInterruptException:
        pass
´´´

### package.xml

```xml
<package format="2">
<name>autoturtle</name>
<version>0.0.0</version>
<description>The autoturtle package, avoids colliding into walls in turtlesim simulator</description>
<!--  One maintainer tag required, multiple allowed, one person per tag  -->
<!--  Example:   -->
<!--  <maintainer email="jane.doe@example.com">Jane Doe</maintainer>  -->
<maintainer email="jgonzalezi@unal.edu.co">Juan Carlos Gonzalez</maintainer>
<!--  One license tag required, multiple allowed, one license per tag  -->
<!--  Commonly used license strings:  -->
<!--    BSD, MIT, Boost Software License, GPLv2, GPLv3, LGPLv2.1, LGPLv3  -->
<license>Unlicense</license>
<!--  Url tags are optional, but multiple are allowed, one per tag  -->
<!--  Optional attribute type can be: website, bugtracker, or repository  -->
<!--  Example:  -->
<!--  <url type="website">http://wiki.ros.org/autoturtle</url>  -->
<!--  Author tags are optional, multiple are allowed, one per tag  -->
<!--  Authors do not have to be maintainers, but could be  -->
<!--  Example:  -->
<!--  <author email="jane.doe@example.com">Jane Doe</author>  -->
<!--  The *depend tags are used to specify dependencies  -->
<!--  Dependencies can be catkin packages or system dependencies  -->
<!--  Examples:  -->
<!--  Use depend as a shortcut for packages that are both build and exec dependencies  -->
<!--    <depend>roscpp</depend>  -->
<!--    Note that this is equivalent to the following:  -->
<!--    <build_depend>roscpp</build_depend>  -->
<!--    <exec_depend>roscpp</exec_depend>  -->
<!--  Use build_depend for packages you need at compile time:  -->
<!--    <build_depend>message_generation</build_depend>  -->
<!--  Use build_export_depend for packages you need in order to build against this package:  -->
<!--    <build_export_depend>message_generation</build_export_depend>  -->
<!--  Use buildtool_depend for build tool packages:  -->
<!--    <buildtool_depend>catkin</buildtool_depend>  -->
<!--  Use exec_depend for packages you need at runtime:  -->
<!--    <exec_depend>message_runtime</exec_depend>  -->
<!--  Use test_depend for packages you need only for testing:  -->
<!--    <test_depend>gtest</test_depend>  -->
<!--  Use doc_depend for packages you need only for building documentation:  -->
<!--    <doc_depend>doxygen</doc_depend>  -->
<buildtool_depend>catkin</buildtool_depend>
<build_depend>beginner_tutorials</build_depend>
<build_depend>roscpp</build_depend>
<build_depend>rospy</build_depend>
<build_depend>std_msgs</build_depend>
<build_export_depend>beginner_tutorials</build_export_depend>
<build_export_depend>roscpp</build_export_depend>
<build_export_depend>rospy</build_export_depend>
<build_export_depend>std_msgs</build_export_depend>
<exec_depend>beginner_tutorials</exec_depend>
<exec_depend>roscpp</exec_depend>
<exec_depend>rospy</exec_depend>
<exec_depend>std_msgs</exec_depend>
<!--  The export tag contains other, unspecified, tags  -->
<export>
<!--  Other tools can request additional information be placed here  -->
</export>
</package> ```

A continuación se muestra un video demostrativo del nodo funcionando.

![](img/lab2_node_demo.gif)